In [2]:
import polars as pl
import psycopg2
import dagster as dg

In [24]:



# class PostgresResource(dg.ConfigurableResource):
#     """Configuration schema for PostgreSQL resource."""

#     host: str
#     port: int
#     database: str
#     user: str
#     password: str
#     chunk_size: int = 300_000

#     def _get_connection(
#         self, context: dg.AssetExecutionContext
#     ) -> psycopg2.extensions.connection:
#         """Creates a psycopg2 connection to PostgreSQL."""

#         message = f"Connecting to PostgreSQL at {self.host}:{self.port} database: {self.database}"
#         context.log.info(message)

#         try:
#             conn = psycopg2.connect(
#                 dbname=self.database,
#                 user=self.user,
#                 host=self.host,
#                 password=self.password,
#                 port=self.port,
#             )
#             return conn
#         except Exception as e:
#             context.log.error(f"Unable to connect to the database: {str(e)}")
#             raise

#     def _get_connection_string(self) -> str:
#         """Constructs a PostgreSQL connection string."""
#         return f"postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}"

#     def __empty_table(
#         self, context: dg.AssetExecutionContext, table_name: str, schema: str
#     ) -> None:
#         """Truncate the specified table."""
#         conn = self._get_connection(context)

#         try:
#             with conn.cursor() as cur:
#                 full_table_name = f"{schema}.{table_name}"
#                 cur.execute(f"TRUNCATE TABLE {full_table_name} CASCADE")
#                 conn.commit()
#         except Exception as e:
#             message = f"Failed to truncate table {schema}.{table_name}: {str(e)}"
#             context.log.error(message)
#             raise
#         finally:
#             conn.close()

#     def get_table_columns(
#         self, context: dg.AssetExecutionContext, table_name: str, schema: str
#     ) -> list:
#         """Get column names of the specified table."""
#         conn = self._get_connection(context)
#         try:
#             with conn.cursor() as cur:
#                 cur.execute(
#                     f"""
#                     SELECT column_name
#                     FROM information_schema.columns
#                     where 
#                         table_schema='{schema}'
#                         and
#                         table_name='{table_name}'
#                     order by ordinal_position;
#                     """
#                 )
#                 columns = [row[0] for row in cur.fetchall()]
#                 return columns
#         except Exception as e:
#             message = f"Failed to get table columns: {schema}.{table_name}: {str(e)}"
#             context.log.error(message)
#             raise
#         finally:
#             conn.close()

#     def _check_table_columns(
#         self, table_name: str, schema: str, df: pl.DataFrame
#     ) -> None:
#         """Check if DataFrame columns match the target table columns."""
#         table_columns = self.get_table_columns(table_name, schema)
#         df_columns = df.columns

#         missing_columns = set(table_columns) - set(df_columns)
#         extra_columns = set(df_columns) - set(table_columns)

#         if missing_columns:
#             raise ValueError(f"Missing columns in DataFrame: {missing_columns}")
#         if extra_columns:
#             raise ValueError(f"Extra columns in DataFrame: {extra_columns}")

#     def load_polars_dataframe(
#         self,
#         context: dg.AssetExecutionContext,
#         df: pl.DataFrame,
#         table_name: str,
#         schema: str,
#     ) -> None:
#         """Load a Polars DataFrame to PostgreSQL in chunks."""
#         try:
#             self.__empty_table(context, table_name, schema)
#             total_chunks = (df.height + self.chunk_size - 1) // self.chunk_size

#             connection_string = self._get_connection_string()

#             for offset in range(0, df.height, self.chunk_size):
#                 message = f"Loading into {schema}.{table_name} chunk {offset // self.chunk_size + 1} of {total_chunks}"
                

#                 batch = df.slice(offset, self.chunk_size)
#                 batch.write_database(
#                     table_name=f"{schema}.{table_name}",
#                     if_table_exists="append",
#                     connection=connection_string,
#                 )
#         except Exception as e:
            
#             raise

#     def get_query_results(
#         self, context: dg.AssetExecutionContext, query: str
#     ) -> pl.DataFrame:
#         """Execute a SQL query and return results as a Polars DataFrame."""
#         conn = self._get_connection(context)
#         try:
#             with conn.cursor() as cur:
#                 cur.execute(query)
#                 columns = [desc[0] for desc in cur.description]
#                 data = cur.fetchall()
#                 return pl.DataFrame(data, schema=columns)
#         except Exception as e:
#             context.log.info(f"Failed to execute query: {str(e)}")
#             raise
#         finally:
#             conn.close()
    
#     def execute_query(self, context: dg.AssetExecutionContext, query: str) -> None:
#         """For executing queries that don't return results."""
#         conn = self._get_connection(context)
#         try:
#             with conn.cursor() as cur:
#                 cur.execute(query)
#             conn.commit()
#         except (Exception, psycopg2.DatabaseError) as e:
#             context.log.info(f"Failed to execute query: {str(e)}")
#             raise
        # finally:
        #     conn.close()






POSTGRES_USER="dagster"
POSTGRES_PASSWORD="4g4WF~E*Zsl217aSErNKu9olVTuP=,HO]dP/FAlH"
POSTGRES_DATABASE="dagster"
POSTGRES_HOST="localhost"
POSTGRES_PORT="5432"
IMDB_SCHEMA="imdb"

uri = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
print(uri)

query = "SELECT * FROM imdb.title_basics LIMIT 10;"
pl.read_database_uri(query, uri) 


postgresql://dagster:4g4WF~E*Zsl217aSErNKu9olVTuP=,HO]dP/FAlH@localhost:5432/dagster


RuntimeError: parse error: invalid port number

In [3]:
title_genres = pl.read_csv("title_genres.csv")
title_genres

id,tconst,genre
i64,str,str
1,"""tt0000001""","""Documentary"""
2,"""tt0000001""","""Short"""
3,"""tt0000002""","""Animation"""
4,"""tt0000002""","""Short"""
5,"""tt0000003""","""Animation"""
…,…,…
96,"""tt0000051""","""Documentary"""
97,"""tt0000051""","""Short"""
98,"""tt0000052""","""Documentary"""


In [10]:
# title_genres.to_dummies(columns=["genre"])
title_genres.select(pl.col("tconst"), pl.col("genre")).to_dummies(columns=["genre"])

tconst,genre_Animation,genre_Comedy,genre_Documentary,genre_Drama,genre_News,genre_Romance,genre_Short,genre_Sport
str,u8,u8,u8,u8,u8,u8,u8,u8
"""tt0000001""",0,0,1,0,0,0,0,0
"""tt0000001""",0,0,0,0,0,0,1,0
"""tt0000002""",1,0,0,0,0,0,0,0
"""tt0000002""",0,0,0,0,0,0,1,0
"""tt0000003""",1,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…
"""tt0000051""",0,0,1,0,0,0,0,0
"""tt0000051""",0,0,0,0,0,0,1,0
"""tt0000052""",0,0,1,0,0,0,0,0


In [21]:
title_genres.with_columns(pl.lit(True).alias("has_genre")).pivot(
    values="has_genre",
    index="tconst",
    columns="genre"
)

/tmp/ipykernel_4028/555398037.py:1: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  title_genres.with_columns(pl.lit(True).alias("has_genre")).pivot(


tconst,Documentary,Short,Animation,Comedy,Romance,Sport,News,Drama
str,bool,bool,bool,bool,bool,bool,bool,bool
"""tt0000001""",true,true,null,null,null,null,null,null
"""tt0000002""",null,true,true,null,null,null,null,null
"""tt0000003""",null,null,true,true,true,null,null,null
"""tt0000004""",null,true,true,null,null,null,null,null
"""tt0000005""",null,true,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…
"""tt0000049""",null,true,null,null,null,true,null,null
"""tt0000050""",true,true,null,null,null,null,null,null
"""tt0000051""",true,true,null,null,null,null,null,null


In [15]:
title_genres.pivot(on="genre", index="tconst")

tconst,Documentary,Short,Animation,Comedy,Romance,Sport,News,Drama
str,i64,i64,i64,i64,i64,i64,i64,i64
"""tt0000001""",1,2,null,null,null,null,null,null
"""tt0000002""",null,4,3,null,null,null,null,null
"""tt0000003""",null,null,5,6,7,null,null,null
"""tt0000004""",null,9,8,null,null,null,null,null
"""tt0000005""",null,10,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…
"""tt0000049""",null,92,null,null,null,93,null,null
"""tt0000050""",94,95,null,null,null,null,null,null
"""tt0000051""",96,97,null,null,null,null,null,null


In [ ]:
watch_status = pl.read_csv("watch_status.csv")
counts = watch_status.select(pl.col("watched").value_counts()).unnest("watched")
counts.filter(pl.col("watched")=="true")["count"].item()
counts.filter(pl.col("watched")=="false")["count"].item()
watch_status["priority"].value_counts().filter(pl.col("priority")=="true")["count"].item()

479

122